# Reseñas de Harbor House Café: del rating al sentimiento escrito

**Pregunta de negocio:** ¿el sentimiento expresado en el texto de las reseñas se alinea con el promedio observado de 4.5/5 estrellas? Usaremos el rating como referencia, no como verdad perfecta del contenido escrito. Este cuaderno cubre la exploración, una estimación automatizada y su revisión cualitativa; no constituye un proyecto de modelado completo.

In [1]:
from pathlib import Path
import sys

import pandas as pd

# El notebook suele ejecutarse con cwd=src; añade src al path para reutilizar app.py.
SRC_DIR = Path.cwd() if Path.cwd().name == "src" else Path.cwd() / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

PROJECT_ROOT = SRC_DIR.parent
DATA_PATH = PROJECT_ROOT / "data/raw/reviews.csv"
reviews = pd.read_csv(DATA_PATH)
print(f"Archivo: {DATA_PATH} | Filas: {reviews.shape[0]:,} | Columnas: {reviews.shape[1]}")
display(reviews.head())

Archivo: /workspaces/aie-pt-3--machine-learning-python-template--MiguelMillan/data/raw/reviews.csv | Filas: 500 | Columnas: 3


,review_id,rating,review_text
0,1,5,Visited Harbor House Café last weekend. Great ...
1,2,5,Tried Harbor House Café after seeing it recomm...
2,3,5,Tried Harbor House Café after seeing it recomm...
3,4,5,Visited Harbor House Café last weekend. The st...
4,5,5,Grabbed a quick coffee at Harbor House Café th...


El CSV cargó correctamente. Antes de inferir cualquier opinión del texto, caractericemos calidad y distribución de la referencia en estrellas.

## 1. Estructura y calidad

Comprobamos esquema, faltantes y consistencia antes de interpretar estrellas o texto.

In [2]:
print("Dimensiones:", reviews.shape)
print("\\nTipos de datos:")
display(reviews.dtypes.rename("dtype").to_frame())
print("\\nValores faltantes por columna:")
display(reviews.isna().sum().rename("faltantes").to_frame())

print(f"IDs únicos: {reviews['review_id'].nunique()} / {len(reviews)}")
print(f"Ratings dentro de 1–5: {reviews['rating'].between(1, 5).all()}")
print(f"Textos vacíos o solo espacios: {reviews['review_text'].str.strip().eq('').sum()}")

Dimensiones: (500, 3)
\nTipos de datos:


,dtype
review_id,int64
rating,int64
review_text,str


\nValores faltantes por columna:


,faltantes
review_id,0
rating,0
review_text,0


IDs únicos: 500 / 500
Ratings dentro de 1–5: True
Textos vacíos o solo espacios: 0


## 2. Ratings y longitud del texto

La distribución muestra qué tan concentrada está la referencia humana; las longitudes ayudan a detectar textos vacíos o casos extremos.

In [3]:
rating_counts = reviews["rating"].value_counts().sort_index()
rating_summary = pd.DataFrame({
    "reseñas": rating_counts,
    "porcentaje": (rating_counts / len(reviews) * 100).round(1),
})
display(rating_summary)
print("\\nEstadísticas de rating:")
display(reviews["rating"].describe().to_frame().T)

text_characters = reviews["review_text"].str.len()
text_words = reviews["review_text"].str.split().str.len()
print("\\nLongitud en caracteres:")
display(text_characters.describe(percentiles=[.25, .5, .75, .9, .95]).to_frame().T)
print("\\nLongitud en palabras:")
display(text_words.describe(percentiles=[.25, .5, .75, .9]).to_frame().T)

,reseñas,porcentaje
rating,,
1,12,2.4
2,18,3.6
3,38,7.6
4,72,14.4
5,360,72.0


\nEstadísticas de rating:


,count,mean,std,min,25%,50%,75%,max
rating,500.0,4.5,0.952794,1.0,4.0,5.0,5.0,5.0


\nLongitud en caracteres:


,count,mean,std,min,25%,50%,75%,90%,95%,max
review_text,500.0,170.17,27.058778,86.0,155.0,173.0,189.0,202.0,210.0,230.0


\nLongitud en palabras:


,count,mean,std,min,25%,50%,75%,90%,max
review_text,500.0,29.104,4.824078,14.0,26.0,29.0,32.0,35.0,42.0


La distribución de estrellas es muy alta; veamos ahora ejemplos reales para detectar mezclas de elogios y quejas que el promedio no muestra.

## 3. Muestra cualitativa del dataset

Revisamos textos con sus ratings para orientar la atención a matices y quejas específicas; no se etiqueta sentimiento a mano en este paso.

In [4]:
sample = reviews.sample(n=6, random_state=42)[["review_id", "rating", "review_text"]]
with pd.option_context("display.max_colwidth", 220):
    display(sample.reset_index(drop=True))

,review_id,rating,review_text
0,362,3,"Stopped by Harbor House Café for the first time. The coffee was okay. Average service, no complaints. It's fine for a quick stop."
1,74,5,"Stopped by Harbor House Café for the first time. The place has such a cozy, welcoming vibe. Every dish was bursting with flavor. Five stars without hesitation."
2,375,5,"Grabbed a quick coffee at Harbor House Café this morning. Service was fast and friendly. The place has such a cozy, welcoming vibe. Reasonably priced for the quality. Already planning my next visit."
3,156,4,"Grabbed a quick coffee at Harbor House Café this morning. The coffee was rich and perfectly brewed. The staff seemed annoyed to be there, but overall a solid experience. It's fine for a quick stop."
4,105,5,Grabbed a quick coffee at Harbor House Café this morning. The staff were warm and attentive. The music and decor set the perfect mood. Great value for what you get. This is now my go-to spot.
5,395,4,"The food was absolutely delicious. We waited 25 minutes just to get water, but overall a solid experience. Might give it another shot."


La inspección respalda conservar textos y registros originales. Con esa limpieza mínima, se define el plan para contrastar sentimiento escrito y estrellas.

## 4. Insights de EDA y plan de preparación

- El promedio de rating es 4.5/5: el 72% de reseñas da 5 estrellas y el 86.4% da 4–5.
- La minoría de 68 reseñas con 1–3 estrellas puede ocultar problemas específicos; conviene priorizar su revisión por temas (servicio, espera y comida).
- Hay 7 textos duplicados (493 textos únicos); validar antes de deduplicar, ya que pueden representar reseñas distintas.

**Limpieza:** no se justifican cambios correctivos: hay 500 filas, ningún faltante ni texto vacío, IDs únicos y ratings válidos de 1 a 5. Conservaremos el texto original y no eliminaremos duplicados automáticamente. La longitud va de 86 a 230 caracteres (mediana 173), sin anomalías evidentes.

**Plan de acción:** medir sentimiento escrito en las 500 reseñas con un clasificador multilingüe de estrellas; comparar su polaridad agregada con el promedio 4.5, inspeccionar desacuerdos y llevar patrones concretos a seguimiento humano. El resultado es una señal exploratoria, no un reemplazo del rating ni una evaluación definitiva del servicio.

**Modelo comprometido:** `nlptown/bert-base-multilingual-uncased-sentiment`, con revisión inmutable `8f6f4e3a8f70be4b65d3a4a8762b6d781cda240d` y paquetes fijados (`transformers==4.52.4`, `torch==2.7.1`). Sus etiquetas 1–5 estrellas se mapean literalmente a estrellas predichas; 1–2 = negativo, 3 = neutral, 4–5 = positivo. `pipeline()` descarga pesos en caché cuando hace falta; estos no se guardan en el repositorio. No se ajusta ni entrena el modelo.

## 5. Inferencia en las 500 reseñas

Cargamos el modelo una sola vez y usamos exactamente la lógica de producción de `src/app.py`. `pipeline()` recibe todas las reseñas en lotes; no se descarga ni recarga por reseña. La salida conserva columnas originales y añade rating predicho, banda y confianza. Los pesos se descargan/cargan desde caché y no se versionan en Git.

In [5]:
from app import MODEL_NAME, load_sentiment_model, predict_reviews

sentiment_model = load_sentiment_model()  # Una sola carga para todo el conjunto.
scored_reviews = predict_reviews(reviews, sentiment_model)

output_path = PROJECT_ROOT / "data/processed/reviews_with_sentiment.csv"
output_path.parent.mkdir(parents=True, exist_ok=True)
scored_reviews.to_csv(output_path, index=False)
print(f"Modelo: {MODEL_NAME}")
print(f"Predicciones: {len(scored_reviews)} | CSV: {output_path}")
display(scored_reviews[["review_id", "rating", "predicted_stars", "sentiment_band", "sentiment_confidence"]].head())

Device set to use cpu


Modelo: nlptown/bert-base-multilingual-uncased-sentiment
Predicciones: 500 | CSV: /workspaces/aie-pt-3--machine-learning-python-template--MiguelMillan/data/processed/reviews_with_sentiment.csv


,review_id,rating,predicted_stars,sentiment_band,sentiment_confidence
0,1,5,5,positive,0.716958
1,2,5,5,positive,0.982220
2,3,5,5,positive,0.662323
3,4,5,5,positive,0.989547
4,5,5,5,positive,0.982912


Ya tenemos 500 predicciones y sus bandas. Resumamos la distribución y comparemos las dos formas de expresar satisfacción: palabras y estrellas.

## 6. Desglose de sentimiento frente a 4.5/5

Comparamos distribución de bandas, promedio de estrellas predichas y calificación humana. Son métricas distintas: no esperamos igualdad exacta entre polaridad por texto y rating.

In [6]:
band_order = ["positive", "neutral", "negative"]
band_counts = scored_reviews["sentiment_band"].value_counts().reindex(band_order, fill_value=0)
band_summary = pd.DataFrame({
    "reviews": band_counts,
    "percent": (band_counts / len(scored_reviews) * 100).round(1),
})
display(band_summary)

human_mean = scored_reviews["rating"].mean()
predicted_mean = scored_reviews["predicted_stars"].mean()
print(f"Media humana: {human_mean:.2f}/5")
print(f"Media predicha (estrellas): {predicted_mean:.2f}/5")
print(f"Diferencia predicha - humana: {predicted_mean - human_mean:+.2f}")
print(f"Ratings humanos 4–5 estrellas: {(scored_reviews['rating'].ge(4).mean() * 100):.1f}%")
print(f"Sentimiento positivo por texto: {(scored_reviews['sentiment_band'].eq('positive').mean() * 100):.1f}%")

,reviews,percent
sentiment_band,,
positive,412,82.4
neutral,41,8.2
negative,47,9.4


Media humana: 4.50/5
Media predicha (estrellas): 4.40/5
Diferencia predicha - humana: -0.10
Ratings humanos 4–5 estrellas: 86.4%
Sentimiento positivo por texto: 82.4%


## 7. Desacuerdos para revisión

Buscamos ratings bajos del modelo frente a ratings humanos altos, y el patrón inverso. Los desacuerdos son candidatos a inspección, no errores confirmados.

In [7]:
false_negative_candidates = scored_reviews[
    scored_reviews["predicted_stars"].le(2) & scored_reviews["rating"].ge(4)
].copy()
false_positive_candidates = scored_reviews[
    scored_reviews["predicted_stars"].ge(4) & scored_reviews["rating"].le(2)
].copy()

print(f"Candidatos a falso negativo (modelo 1–2, humano 4–5): {len(false_negative_candidates)}")
print(f"Desacuerdo inverso (modelo 4–5, humano 1–2): {len(false_positive_candidates)}")
with pd.option_context("display.max_colwidth", 240):
    display(false_negative_candidates[["review_id", "rating", "predicted_stars", "sentiment_band", "review_text"]].head(20))

Candidatos a falso negativo (modelo 1–2, humano 4–5): 16
Desacuerdo inverso (modelo 4–5, humano 1–2): 0


,review_id,rating,predicted_stars,sentiment_band,review_text
13,14,4,1,negative,Tried Harbor House Café after seeing it recommended online. Every dish was bursting with flavor. The team clearly cares about the customers. Already planning my next visit.
14,15,5,1,negative,"Every dish was bursting with flavor. The place has such a cozy, welcoming vibe. Already planning my next visit."
72,73,5,1,negative,Stopped by Harbor House Café for the first time. Service was fast and friendly. Every dish was bursting with flavor. Worth every penny. Already planning my next visit.
86,87,5,1,negative,Tried Harbor House Café after seeing it recommended online. The seasonal menu never disappoints. They went out of their way to accommodate my allergy. Will definitely be back!
137,138,5,2,negative,"Tried Harbor House Café after seeing it recommended online. They went out of their way to accommodate my allergy. The place has such a cozy, welcoming vibe. Already planning my next visit."
199,200,5,2,negative,Stopped by Harbor House Café for the first time. The music and decor set the perfect mood. Every dish was bursting with flavor. Already planning my next visit.
203,204,5,1,negative,"Came to Harbor House Café for a birthday brunch. Every dish was bursting with flavor. The cashier was rude when i asked about the menu, but honestly the food made up for it. Already planning my next visit."
253,254,5,1,negative,Stopped by Harbor House Café for the first time. They went out of their way to accommodate my allergy. Great spot to work or catch up with friends. Will definitely be back!
269,270,5,1,negative,Stopped by Harbor House Café for the first time. Our server remembered our names by the second visit. Every dish was bursting with flavor. Great value for what you get. Will definitely be back!
321,322,5,1,negative,"Every dish was bursting with flavor. The staff seemed annoyed to be there, but honestly the food made up for it. Will definitely be back!"


El resumen agregado oculta casos discordantes; inspeccionemos primero los posibles falsos negativos y falsos positivos con sus textos.

## 8. Auditoría manual de 20 reseñas

Tomamos una muestra reproducible y estratificada de 18 (6 por banda). Leemos texto y predicción juntos; se anotan ejemplos y se marca si parecen razonables o revisables. No es una validación estadística.

In [8]:
# Seis por banda: 18 reseñas, selección reproducible. Las notas son lectura manual del texto.
manual_sample = scored_reviews.groupby("sentiment_band", group_keys=False).sample(n=6, random_state=19)
manual_sample = manual_sample[["review_id", "rating", "predicted_stars", "sentiment_band", "sentiment_confidence", "review_text"]].copy()
manual_notes = {
    204: "Probable falso negativo: elogios claros a comida y visita futura, con una queja localizada sobre la cajera.",
    342: "Negativo tiene sentido: café quemado, falta de atención y ambiente caótico.",
    195: "Negativo tiene sentido: comida sosa/cara y pedido equivocado.",
    190: "Negativo tiene sentido: porciones pequeñas para el precio y nadie atendió la mesa.",
    115: "Negativo tiene sentido: pastel rancio y espera larga para recibir agua.",
    314: "Negativo tiene sentido: huevos fríos, falta de seguimiento y duda de volver.",
    239: "Neutral tiene sentido: menú repetitivo y servicio lento, sin rechazo fuerte.",
    233: "Neutral tiene sentido: café correcto, servicio algo lento y posibilidad de volver.",
    80: "Neutral tiene sentido: experiencia aceptable pero poco memorable.",
    161: "Neutral tiene sentido: menú algo repetitivo y servicio lento, con disposición a regresar.",
    253: "Positivo al leerlo, aunque con reserva por servicio promedio; neutral parece conservador.",
    464: "Probable infravaloración: comida excelente, experiencia sólida y regreso previsto pese a servicio sin brillo.",
    274: "Positivo tiene sentido: comida, ambiente y cierre expresan satisfacción fuerte.",
    205: "Positivo tiene sentido: equipo atento y recomendación explícita.",
    296: "Positivo tiene sentido: ambiente acogedor, servicio rápido y lo considera su lugar habitual.",
    166: "Positivo razonable: mención negativa del personal queda mitigada por valoración global y cinco estrellas explícitas.",
    118: "Positivo tiene sentido: buen servicio, ambiente, valor y deseo de volver.",
    12: "Positivo tiene sentido: pasteles, servicio y valor elogiados; expresa intención de volver.",
}
manual_sample["manual_read"] = manual_sample["review_id"].map(manual_notes).fillna(
    "Lectura general consistente con la banda predicha; no destaca un error claro."
)
manual_sample["model_looks_wrong?"] = manual_sample["review_id"].isin([204, 253, 464]).map({True: "Revisar", False: "No evidente"})
manual_sample = manual_sample.reset_index(drop=True)
with pd.option_context("display.max_colwidth", 240):
    display(manual_sample)

,review_id,rating,predicted_stars,sentiment_band,sentiment_confidence,review_text,manual_read,model_looks_wrong?
0,342,1,2,negative,0.508174,Regular customer at Harbor House Café here. The coffee tasted burnt. Nobody checked on us the entire meal. The place felt cramped and chaotic. Expected a lot more given the reviews.,"Negativo tiene sentido: café quemado, falta de atención y ambiente caótico.",No evidente
1,195,2,1,negative,0.476999,Grabbed a quick coffee at Harbor House Café this morning. The food was bland and overpriced. Our server got our order wrong twice. Probably won't recommend it to friends.,Negativo tiene sentido: comida sosa/cara y pedido equivocado.,No evidente
2,190,1,2,negative,0.562111,Stopped by Harbor House Café for the first time. The portions were way too small for the price. Nobody checked on us the entire meal. Disappointing experience overall.,Negativo tiene sentido: porciones pequeñas para el precio y nadie atendió la mesa.,No evidente
3,115,2,2,negative,0.624875,Grabbed a quick coffee at Harbor House Café this morning. The pastry was stale. We waited 25 minutes just to get water. Disappointing experience overall.,Negativo tiene sentido: pastel rancio y espera larga para recibir agua.,No evidente
4,204,5,1,negative,0.349168,"Came to Harbor House Café for a birthday brunch. Every dish was bursting with flavor. The cashier was rude when i asked about the menu, but honestly the food made up for it. Already planning my next visit.","Probable falso negativo: elogios claros a comida y visita futura, con una queja localizada sobre la cajera.",Revisar
5,314,1,1,negative,0.418783,Regular customer at Harbor House Café here. My eggs came out cold. Nobody checked on us the entire meal. Not sure I'll be returning.,"Negativo tiene sentido: huevos fríos, falta de seguimiento y duda de volver.",No evidente
6,464,4,3,neutral,0.472534,"Regular customer at Harbor House Café here. Every dish was bursting with flavor. Service was fine, nothing memorable, but overall a solid experience. Already planning my next visit.","Probable infravaloración: comida excelente, experiencia sólida y regreso previsto pese a servicio sin brillo.",Revisar
7,239,3,3,neutral,0.524050,Came to Harbor House Café for a birthday brunch. The menu is fine but a bit repetitive. The staff were polite but a bit slow. It's fine for a quick stop.,"Neutral tiene sentido: menú repetitivo y servicio lento, sin rechazo fuerte.",No evidente
8,233,3,3,neutral,0.689949,Stopped by Harbor House Café for the first time. The coffee was okay. The staff were polite but a bit slow. Might give it another shot.,"Neutral tiene sentido: café correcto, servicio algo lento y posibilidad de volver.",No evidente
9,80,3,3,neutral,0.737994,"Regular customer at Harbor House Café here. The coffee was okay. Service was fine, nothing memorable. Wouldn't go out of my way, but wouldn't avoid it either.",Neutral tiene sentido: experiencia aceptable pero poco memorable.,No evidente


## 9. Cierre para la account manager

El texto y las estrellas apuntan a una percepción global favorable, pero no son intercambiables: el modelo clasificó 82.4% como positivo frente a 86.4% de ratings humanos de 4–5 estrellas. La media predicha fue 4.40/5, 0.10 por debajo de 4.50. Los grupos concuerdan en dirección general, aunque el clasificador es algo más conservador.

Hay 16 posibles falsos negativos claros por el criterio rating humano 4–5 / modelo 1–2, y ninguno en el sentido inverso. Las reseñas 14 y 15, por ejemplo, contienen elogios claros pero recibieron 1 estrella predicha. **Hipótesis:** el clasificador nlptown se entrenó para reseñas de productos; en reseñas de servicios de café puede sobrerreaccionar a palabras locales negativas ("rude", "annoyed") o no integrar bien mitigaciones como "the food made up for it" y señales de regreso futuro. Las frases formulaicas pueden añadir otra fuente de error. Es una hipótesis plausible que requiere más auditoría, no una causa demostrada.

En la auditoría estratificada de 18, la mayoría de los ejemplos son coherentes con la banda; los casos 204 (modelo negativo con elogios y queja localizada), 253 y 464 (modelo neutral pese al tono positivo general) merecen revisión. Los casos se documentan en la tabla. La selección no es una validación estadística.

**Siguiente paso recomendado:** compartir los casos de desacuerdo con el equipo de cuenta para comprobar temas y contexto de origen. Tratar las etiquetas como priorización exploratoria y no como evidencia para decisiones sobre clientes o personal; si el uso va a ser operativo, primero ampliar y documentar la auditoría humana.